# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PaNavar369/Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My baseline rule prioritizes pages using two observable search signals: content staleness and CTR relative to the expected CTR for the page's position tier. Staleness is measured using days since the last update, while the CTR signal compares a page's observed CTR with the median CTR for pages in the same position tier using the available historical observation. The purpose is to create a simple, transparent baseline that can be inspected and later compared with an ML model.

The score combines the two signals into one priority score. Higher staleness increases the score, and a CTR that is substantially below the position-tier benchmark also increases the score.

The rule uses one reason code for each recommendation:

STALE_CONTENT — the staleness component is the stronger reason for the recommendation.
CTR_BELOW_POSITION_BENCHMARK — the CTR deficit is the stronger reason.

LOW_PRIORITY — neither signal is strong enough to justify a high-priority action.

The action labels are REFRESH, CTR_FIX, and MONITOR. This is a decision-support baseline, not a claim that either signal causes ranking movement.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:


import numpy as np
import pandas as pd
import os


df = pd.read_csv(
    "https://raw.githubusercontent.com/PaNavar369/Internship/main/data/raw/content_refresh_anonymized.csv"
)
df["stale_score"] = (
    df["days_since_last_update"] / 180
).clip(0, 1)


benchmark_df = df[df["impressions_90d"] >= 100].copy()

position_benchmark = (
    benchmark_df
    .groupby("position_tier")["ctr"]
    .median()
)

# Map the benchmark back to every page.
df["position_ctr_benchmark"] = df["position_tier"].map(
    position_benchmark
)


df["ctr_deficit"] = np.where(
    (df["impressions_90d"] >= 100) &
    (df["position_ctr_benchmark"] > 0),
    (
        1 -
        df["ctr"] / df["position_ctr_benchmark"]
    ).clip(0, 1),
    0
)

# Combine the two signals into one transparent score.
df["action_score"] = (
    100 * (
        0.55 * df["stale_score"] +
        0.45 * df["ctr_deficit"]
    )
)

# Select ONE reason code for each page.
df["reason_code"] = np.select(
    [
        (df["action_score"] >= 60) &
        (df["stale_score"] >= df["ctr_deficit"]),

        (df["action_score"] >= 60) &
        (df["ctr_deficit"] > df["stale_score"])
    ],
    [
        "STALE_CONTENT",
        "CTR_BELOW_POSITION_BENCHMARK"
    ],
    default="LOW_PRIORITY"
)

# Assign the corresponding action label.
df["action"] = np.select(
    [
        df["reason_code"] == "STALE_CONTENT",
        df["reason_code"] == "CTR_BELOW_POSITION_BENCHMARK"
    ],
    [
        "REFRESH",
        "CTR_FIX"
    ],
    default="MONITOR"
)

# Rank all pages from highest to lowest priority.
ranked = (
    df.sort_values(
        ["action_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked["rank"] = ranked.index + 1

# Create the required output.
baseline_queue = ranked[
    [
        "rank",
        "content_id",
        "action_score",
        "reason_code",
        "action"
    ]
].copy()

os.makedirs("work/outputs", exist_ok=True)

# Write the ranked queue
output_path = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(
    output_path,
    index=False
)

print(f"Ranked queue written to: {output_path}")
print(f"Rows written: {len(baseline_queue):,}")

display(baseline_queue.head(10))

Ranked queue written to: work/outputs/baseline_action_score.csv
Rows written: 30,000


,rank,content_id,action_score,reason_code,action
0,1,content_b16bd7307b39,100.0,STALE_CONTENT,REFRESH
1,2,content_074ba6ead17b,100.0,STALE_CONTENT,REFRESH
2,3,content_fd16e3475c29,100.0,STALE_CONTENT,REFRESH
3,4,content_4f241bad48a3,100.0,STALE_CONTENT,REFRESH
4,5,content_ea41fe5cf292,100.0,STALE_CONTENT,REFRESH
5,6,content_958a46db26bd,100.0,STALE_CONTENT,REFRESH
6,7,content_02b0d6e30129,100.0,STALE_CONTENT,REFRESH
7,8,content_f488400fca67,100.0,STALE_CONTENT,REFRESH
8,9,content_b6e4581523ed,100.0,STALE_CONTENT,REFRESH
9,10,content_ab27c30d81f4,100.0,STALE_CONTENT,REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
# Top-20 review

top20 = ranked.head(20).copy()

def confidence_note(row):
    if row["impressions_90d"] >= 1000:
        return "Higher confidence: substantial observed search visibility."
    elif row["impressions_90d"] >= 100:
        return "Moderate confidence: enough impressions for the CTR benchmark."
    else:
        return "Lower confidence: limited search visibility."

def wrong_if(row):
    if row["reason_code"] == "STALE_CONTENT":
        return (
            "Wrong if the page is intentionally evergreen or was recently "
            "updated outside the measured field."
        )
    elif row["reason_code"] == "CTR_BELOW_POSITION_BENCHMARK":
        return (
            "Wrong if the low CTR is explained by query mix, SERP features, "
            "brand effects, or another factor not captured by this baseline."
        )
    else:
        return (
            "Wrong if an important signal missing from this simple baseline "
            "should have changed the priority."
        )

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_if,
    axis=1
)

top20_review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "action_score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(top20_review)

,rank,content_id,action,reason_code,action_score,confidence_note,what_would_make_it_wrong
0,1,content_b16bd7307b39,REFRESH,STALE_CONTENT,100.000000,Higher confidence: substantial observed search...,Wrong if the page is intentionally evergreen o...
1,2,content_074ba6ead17b,REFRESH,STALE_CONTENT,100.000000,Moderate confidence: enough impressions for th...,Wrong if the page is intentionally evergreen o...
2,3,content_fd16e3475c29,REFRESH,STALE_CONTENT,100.000000,Moderate confidence: enough impressions for th...,Wrong if the page is intentionally evergreen o...
3,4,content_4f241bad48a3,REFRESH,STALE_CONTENT,100.000000,Moderate confidence: enough impressions for th...,Wrong if the page is intentionally evergreen o...
4,5,content_ea41fe5cf292,REFRESH,STALE_CONTENT,100.000000,Moderate confidence: enough impressions for th...,Wrong if the page is intentionally evergreen o...
5,6,content_958a46db26bd,REFRESH,STALE_CONTENT,100.000000,Moderate confidence: enough impressions for th...,Wrong if the page is intentionally evergreen o...
6,7,content_02b0d6e30129,REFRESH,STALE_CONTENT,100.000000,Moderate confidence: enough impressions for th...,Wrong if the page is intentionally evergreen o...
7,8,content_f488400fca67,REFRESH,STALE_CONTENT,100.000000,Moderate confidence: enough impressions for th...,Wrong if the page is intentionally evergreen o...
8,9,content_b6e4581523ed,REFRESH,STALE_CONTENT,100.000000,Moderate confidence: enough impressions for th...,Wrong if the page is intentionally evergreen o...
9,10,content_ab27c30d81f4,REFRESH,STALE_CONTENT,100.000000,Moderate confidence: enough impressions for th...,Wrong if the page is intentionally evergreen o...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
# --------------------------------
# Weak picks
# --------------------------------

weak_picks = ranked[
    (ranked["action_score"] >= 60) &
    (ranked["impressions_90d"] < 100)
].head(10)

print("Potential weak picks:")
display(
    weak_picks[
        [
            "content_id",
            "impressions_90d",
            "days_since_last_update",
            "ctr",
            "position_tier",
            "action_score",
            "reason_code",
            "action"
        ]
    ]
)


# --------------------------------
# Leakage check
# --------------------------------

score_inputs = [
    "days_since_last_update",
    "ctr",
    "position_tier",
    "impressions_90d"
]

forbidden_outcome_fields = [
    "trend_direction",
    "trend_pct"
]

print("\nLeakage check:")

for field in forbidden_outcome_fields:
    print(
        f"{field}: "
        f"present={field in df.columns}, "
        f"used_as_score_input={field in score_inputs}"
    )

assert "trend_direction" not in score_inputs
assert "trend_pct" not in score_inputs

print("\nPASS: outcome fields are not used in the baseline score.")
print("PASS: no future-window outcome is used in scoring.")
print("PASS: no product flag is used as a scoring input.")

Potential weak picks:


,content_id,impressions_90d,days_since_last_update,ctr,position_tier,action_score,reason_code,action



Leakage check:
trend_direction: present=True, used_as_score_input=False
trend_pct: present=True, used_as_score_input=False

PASS: outcome fields are not used in the baseline score.
PASS: no future-window outcome is used in scoring.
PASS: no product flag is used as a scoring input.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.